# Notebook 05 — Final Load Prep & KPI Export
**Project:** Credit Risk & Loan Default Analysis  
**Objective:** Compute all KPIs and export final aggregated datasets for Tableau dashboard.

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/processed/loans_cleaned.csv', low_memory=False)
df['issue_d'] = pd.to_datetime(df['issue_d'], errors='coerce')
print(f'Loaded: {df.shape}')

OUTPUT_DIR = '../data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

Loaded: (1348099, 32)


## KPI 1 — Overall Portfolio Summary

In [2]:
portfolio_kpis = pd.DataFrame([{
    'total_loans': len(df),
    'total_loan_volume': df['loan_amnt'].sum(),
    'overall_default_rate': df['default_flag'].mean(),
    'avg_loan_amount': df['loan_amnt'].mean(),
    'avg_interest_rate': df['int_rate'].mean(),
    'avg_dti': df['dti'].mean(),
    'avg_fico': df['fico_avg'].mean(),
    'total_defaulted_loans': df['default_flag'].sum(),
    'total_defaulted_amount': df[df['default_flag'] == 1]['loan_amnt'].sum()
}])

portfolio_kpis.to_csv(f'{OUTPUT_DIR}kpi_portfolio_summary.csv', index=False)
print('Saved: kpi_portfolio_summary.csv')
portfolio_kpis.T

Saved: kpi_portfolio_summary.csv


,0
total_loans,1.348099e+06
total_loan_volume,1.942476e+10
overall_default_rate,1.998073e-01
avg_loan_amount,1.440900e+04
avg_interest_rate,1.324156e+01
avg_dti,1.811976e+01
avg_fico,6.981623e+02
total_defaulted_loans,2.693600e+05
total_defaulted_amount,4.187960e+09


## KPI 2 — Default Rate by Grade

In [3]:
grade_kpis = df.groupby('grade').agg(
    loan_count=('loan_amnt', 'count'),
    total_volume=('loan_amnt', 'sum'),
    default_rate=('default_flag', 'mean'),
    defaulted_loans=('default_flag', 'sum'),
    avg_int_rate=('int_rate', 'mean'),
    avg_fico=('fico_avg', 'mean'),
    avg_dti=('dti', 'mean')
).reset_index().sort_values('grade')

grade_kpis.to_csv(f'{OUTPUT_DIR}kpi_by_grade.csv', index=False)
print('Saved: kpi_by_grade.csv')
grade_kpis

Saved: kpi_by_grade.csv


,grade,loan_count,total_volume,default_rate,defaulted_loans,avg_int_rate,avg_fico,avg_dti
0,A,235193,3.266594e+09,0.060435,14214,7.113456,729.529926,15.542113
1,B,393102,5.202049e+09,0.133963,52661,10.679211,699.142513,17.265389
2,C,382323,5.421290e+09,0.224431,85805,14.019234,689.982558,18.742632
3,D,201657,3.075050e+09,0.303803,61264,17.710482,685.262396,19.934916
4,E,94192,1.654585e+09,0.384311,36199,21.106565,683.960198,20.540057
5,F,32306,6.148668e+08,0.451464,14585,24.875708,681.794945,20.719068
6,G,9326,1.903213e+08,0.496676,4632,27.542421,680.188934,20.904798


## KPI 3 — Default Rate by Year

In [4]:
yearly_kpis = df.groupby('issue_year').agg(
    loan_count=('loan_amnt', 'count'),
    total_volume=('loan_amnt', 'sum'),
    default_rate=('default_flag', 'mean'),
    avg_int_rate=('int_rate', 'mean'),
    avg_loan_amnt=('loan_amnt', 'mean')
).reset_index()

yearly_kpis.to_csv(f'{OUTPUT_DIR}kpi_by_year.csv', index=False)
print('Saved: kpi_by_year.csv')
yearly_kpis

Saved: kpi_by_year.csv


,issue_year,loan_count,total_volume,default_rate,avg_int_rate,avg_loan_amnt
0,2007,603,4.977475e+06,0.262023,11.825108,8254.519071
1,2008,2393,2.111925e+07,0.207271,12.061964,8825.428333
2,2009,5281,5.192825e+07,0.136906,12.437247,9833.033516
3,2010,12537,1.319926e+08,0.140145,11.985268,10528.240408
4,2011,21721,2.616838e+08,0.151789,12.223365,12047.503568
5,2012,53367,7.184110e+08,0.161973,13.637787,13461.709015
6,2013,134804,1.982613e+09,0.155960,14.531628,14707.375152
7,2014,223103,3.253490e+09,0.184498,13.656289,14582.903412
8,2015,375546,5.498601e+09,0.201850,12.385206,14641.618204
9,2016,293105,4.240362e+09,0.232859,13.086688,14467.039457


## KPI 4 — Default Rate by State

In [5]:
state_kpis = df.groupby('addr_state').agg(
    loan_count=('loan_amnt', 'count'),
    total_volume=('loan_amnt', 'sum'),
    default_rate=('default_flag', 'mean'),
    avg_int_rate=('int_rate', 'mean')
).reset_index().sort_values('default_rate', ascending=False)

state_kpis.to_csv(f'{OUTPUT_DIR}kpi_by_state.csv', index=False)
print('Saved: kpi_by_state.csv')
state_kpis.head(10)

Saved: kpi_by_state.csv


,addr_state,loan_count,total_volume,default_rate,avg_int_rate
25,MS,6595,9.218970e+07,0.261107,13.518267
29,NE,3592,4.793662e+07,0.252227,13.499190
2,AR,10062,1.357667e+08,0.241105,13.504411
1,AL,16645,2.321572e+08,0.236347,13.727260
36,OK,12298,1.752447e+08,0.234672,13.418118
18,LA,15524,2.211876e+08,0.231770,13.352843
34,NY,110097,1.573630e+09,0.220506,13.379980
33,NV,20296,2.793486e+08,0.219698,13.399076
9,FL,95843,1.315089e+09,0.215018,13.320312
15,IN,21726,3.047970e+08,0.214305,13.387251


## KPI 5 — Default Rate by Purpose

In [6]:
purpose_kpis = df.groupby('purpose').agg(
    loan_count=('loan_amnt', 'count'),
    total_volume=('loan_amnt', 'sum'),
    default_rate=('default_flag', 'mean'),
    avg_loan_amnt=('loan_amnt', 'mean'),
    avg_int_rate=('int_rate', 'mean')
).reset_index().sort_values('default_rate', ascending=False)

purpose_kpis.to_csv(f'{OUTPUT_DIR}kpi_by_purpose.csv', index=False)
print('Saved: kpi_by_purpose.csv')
purpose_kpis

Saved: kpi_by_purpose.csv


,purpose,loan_count,total_volume,default_rate,avg_loan_amnt,avg_int_rate
11,SMALL_BUSINESS,15577,2.434776e+08,0.298645,15630.583553,15.929020
10,RENEWABLE_ENERGY,936,9.364050e+06,0.237179,10004.326923,15.187137
8,MOVING,9526,7.481668e+07,0.233991,7853.944468,15.146970
5,HOUSE,7298,1.120675e+08,0.219101,15355.916004,15.409837
7,MEDICAL,15614,1.403950e+08,0.218458,8991.608492,14.012430
2,DEBT_CONSOLIDATION,781442,1.189752e+10,0.211567,15225.078861,13.624181
9,OTHER,78301,7.683342e+08,0.210827,9812.571998,14.583420
3,EDUCATIONAL,423,2.798600e+06,0.208038,6616.075650,12.120142
12,VACATION,9084,5.622632e+07,0.191986,6189.599846,13.698224
6,MAJOR_PURCHASE,29550,3.491919e+08,0.186058,11816.983926,12.740785


## Final Check — All Exports

In [7]:
import os
files = os.listdir(OUTPUT_DIR)
print('Files in data/processed/:')
for f in sorted(files):
    size = os.path.getsize(f'{OUTPUT_DIR}{f}')
    print(f'  {f} ({size:,} bytes)')

print('\nFinal load prep complete. Ready for Tableau.')

Files in data/processed/:
  kpi_by_grade.csv (799 bytes)
  kpi_by_purpose.csv (1,269 bytes)
  kpi_by_state.csv (3,062 bytes)
  kpi_by_year.csv (1,039 bytes)
  kpi_portfolio_summary.csv (284 bytes)
  loans_cleaned.csv (277,984,432 bytes)

Final load prep complete. Ready for Tableau.
